# Reproducing Table 4

Adaptive CAPTCHA orchestration via reinforcement learning.

Every policy in the paper is evaluated here from the checkpoints that produced the
published numbers. All the machinery lives in the `rlcaptcha` package; this notebook
only wires it together.

Read `README.md` first if you care about *why* some policies use different reward
settings -- the published runs were not quite like-for-like, and that is preserved
deliberately rather than silently fixed.

In [ ]:
%load_ext autoreload
%autoreload 2

from rlcaptcha import *

humans, bots = load_sessions()
print(f'{len(humans)} human sessions, {len(bots)} bot sessions')

## 1. Behavioural scorer

The LSTM returns P(bot): **high means bot-like**. The manuscript's Humanity Score H
is `1 - score`.

Scores are memoised on `(recording, chunk index)`. Every simulated user is a copy of
one of 73 recordings and the only thing separating two copies is a one-pixel rigid
translation, so ~80k inferences collapse to at most 876. See the `scoring` module
docstring for the full argument, and `verify_cache` below for the measured error.

In [ ]:
scorer = BotScorer()
cache = ScoreCache(scorer).precompute(humans, bots)

In [ ]:
# How much does the cache actually approximate away?
verify_cache(scorer, humans + bots, n=25)

## 2. The six policies

Each is paired with the reward configuration its published run used.

In [ ]:
policies = [
    (LinUCBPolicy(),          BANDIT_REWARDS),
    (ThompsonPolicy(),        BANDIT_REWARDS),
    (DQNPolicy(),             DQN_REWARDS),
    (DQNAblationPolicy(),     DQN_REWARDS),
    (SingleThresholdPolicy(), DQN_REWARDS),
    (MultiThresholdPolicy(),  DQN_REWARDS),
]

for p, _ in policies:
    print(f'{p.name:26s} score={p.needs_bot_score!s:5s} '
          f'acts_when_exhausted={p.acts_when_exhausted!s:5s} '
          f'last_threat_init={p.initial_last_threat}')

## 3. Run every simulation

100 humans against 0 / 20 / 100 / 200 / 500 / 1000 bots, for each policy.
Pass `seed=` for a deterministic run.

In [ ]:
sweeps = {}
for policy, rewards in policies:
    sweeps[policy.name] = run_sweep(
        policy, humans, bots, cache=cache, rewards=rewards, seed=0
    )

## 4. Table 4

In [ ]:
df = table(sweeps)
df.pivot(index='Bots', columns='Policy',
         values=['Remaining Humans', 'Remaining Bots'])

In [ ]:
df.pivot(index='Bots', columns='Policy', values=['DI', 'BOS', 'SP-F1', 'SI-F1'])

### Averages

As in the paper, the spread is the standard deviation **across the six simulation
types**, not across repeated seeds.

In [ ]:
import pandas as pd

rows = {}
for name, sweep in sweeps.items():
    avg = summarise(sweep)['average']
    rows[name] = {k: f"{v['mean']:.3f} ± {v['std']:.3f}" for k, v in avg.items()}
pd.DataFrame(rows).T

## 5. Population dynamics

Humans and bots remaining at each timestep, per policy and bot volume.

In [ ]:
import matplotlib.pyplot as plt

names = list(sweeps)
sims = list(EVAL_BOT_COUNTS)
fig, axes = plt.subplots(len(sims), len(names),
                         figsize=(3.2 * len(names), 2.6 * len(sims)),
                         sharex=True, sharey=True)

for col, name in enumerate(names):
    for row, nb in enumerate(sims):
        ax = axes[row, col]
        r = sweeps[name][nb]
        ax.plot(r.timesteps, r.human_counts, color='tab:blue', lw=1.8, label='Humans')
        ax.plot(r.timesteps, r.bot_counts, color='tab:red', ls='--', lw=1.8, label='Bots')
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(name, fontsize=10)
        if col == 0:
            ax.set_ylabel(str(nb) + ' Bots\n\nUsers')
        if row == len(sims) - 1:
            ax.set_xlabel('Timestep')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('results/population_dynamics_grid.pdf', dpi=300)
plt.show()

## 6. Figure 4 -- ablation study

DQN against the three ablation variants across the four metrics. This figure had no
source before; it was previously a chart in the results spreadsheet.

In [ ]:
ablation = ['DQN', 'DQN without H-Score',
            'Static Single-Threshold', 'Static Multi-Threshold']
metrics = ['DI', 'BOS', 'SP-F1', 'SI-F1']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric in zip(axes.ravel(), metrics):
    for name in ablation:
        sub = df[df.Policy == name].sort_values('Bots')
        vals = sub[metric].to_numpy()
        line, = ax.plot(sub.Bots, vals, marker='o', label=name)
        ax.axhline(vals.mean(), color=line.get_color(), ls=':', alpha=0.6)
        ax.fill_between(sub.Bots, vals.mean() - vals.std(), vals.mean() + vals.std(),
                        color=line.get_color(), alpha=0.08)
    ax.set_xscale('symlog')
    ax.set_xticks(EVAL_BOT_COUNTS)
    ax.set_xticklabels(EVAL_BOT_COUNTS)
    ax.set_xlabel('Number of bots')
    ax.set_title(metric)
    ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=9)
plt.tight_layout()
plt.savefig('results/ablation_study.pdf', dpi=300)
plt.show()

## 7. Check against the published values

`validate_against_paper.py` runs every policy over several seeds and prints the
published number next to the reproduced mean +/- std. Run it from a shell:

```bash
python validate_against_paper.py
```